# 04 · Pretrained Backbone + Cross-Disaster Evaluation  ·  *Role D*

**Goal (Part 2, model 2 + deployment):** transfer-learning model on a pretrained backbone,
then the **held-out-disaster** evaluation, scratch-vs-pretrained comparison, and damage map.

In [ ]:
import sys, os; sys.path.append(os.path.abspath('..'))
from src import config, data_utils, models, train_utils
import torch, numpy as np
from torch.utils.data import DataLoader
from pathlib import Path
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

### Model — siamese with a pretrained ResNet18 backbone (shared, fine-tuned)
`models.SiamesePretrainedCNN` wraps a torchvision ResNet18 (ImageNet weights), shared across
both branches, with a small classification head. Freeze early layers, fine-tune the rest.

In [ ]:
model = models.SiamesePretrainedCNN(num_classes=config.NUM_CLASSES, backbone='resnet18',
                                    freeze_until=6).to(device)
print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6, 'M')

### Train on seen disasters

In [ ]:
train_dl = DataLoader(data_utils.DamageDataset('train', augment=True),  batch_size=config.BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(data_utils.DamageDataset('val',   augment=False), batch_size=config.BATCH_SIZE)
w_path = Path(config.DATA_DIR)/'processed'/'class_weights.npy'
class_weights = torch.tensor(np.load(w_path), dtype=torch.float32) if w_path.exists() else None
history = train_utils.train_model(model, train_dl, val_dl, device, epochs=config.EPOCHS,
                                  class_weights=class_weights, ckpt_path='../checkpoints/pretrained.pt')
train_utils.plot_history(history)

## Held-out-disaster evaluation (the key result)
Test set = `config.HELDOUT_DISASTER` only. Load **both** trained models and compare:
accuracy, per-class precision/recall/F1, confusion matrix — on **seen** (val) vs **held-out** (test).

In [ ]:
test_dl = DataLoader(data_utils.DamageDataset('test', augment=False), batch_size=config.BATCH_SIZE)

scratch = models.SiameseScratchCNN(config.NUM_CLASSES).to(device)
scratch.load_state_dict(torch.load('../checkpoints/scratch.pt', map_location=device))

print('=== FROM-SCRATCH on held-out disaster ===')
train_utils.evaluate(scratch, test_dl, device, class_names=config.DAMAGE_CLASSES)
print('=== PRETRAINED on held-out disaster ===')
train_utils.evaluate(model, test_dl, device, class_names=config.DAMAGE_CLASSES)

### Comparison table & discussion
TODO(Role D): build a small DataFrame: rows = {scratch, pretrained} × {seen, held-out},
columns = accuracy, macro-F1, destroyed-recall. Discuss the transfer gap.

**Optional ablation:** add a few labeled held-out patches to training (few-shot) and
measure how much the gap closes.

In [ ]:
# TODO(Role D): comparison DataFrame from the evaluate() outputs.

## Deployment — damage map for a held-out event
Slide patches over a full held-out tile, predict per building, and render a colored
damage map (green=no-damage … red=destroyed) over the post image.

In [ ]:
# TODO(Role D): tile-level inference -> colored overlay.
# 1. pick a held-out event tile; 2. crop all building patches;
# 3. predict with the better model; 4. draw polygons colored by predicted class.
# Discuss where predictions are reliable vs fail.